In [ ]:
import shaker
import MDAnalysis as md
import nglview as nv

## Generating type 3 virtual sites. This tutorial is a WIP.

Here we will use `shaker` to construct a [type 3 virtual-site](https://manual.gromacs.org/documentation/current/reference-manual/functions/interaction-methods.html#virtual-interaction-sites) representation that captures a molecules average structure. This greatly facilitates the parameterization of rigid molecules, such as cholesterol.

The workflow consists of the following steps:
* **Align all molecules in the trajectory to a common reference:**
  Text
* **Compute an average structure:**
  Text
* **Define a 3 bead frame:**
  Text
* **Construct the remaining beads as virtual sites:**
  The positions of the remaining beads are expressed as virtual sites whose coordinates depend on the frame beads.

In [ ]:
author  = 'Luis Borges-Araujo'
itp_header = [
    f"; Parameterised by {author} @ ENS de Lyon, 2026.\n"
    "; Molecular name: \n",
    "; SMILES: \n",
]


## AA reference sims (already PBC treated and removed solvent.)
GRO = '../AA_references/ComplexMembrane/pbc.gro'
XTC = '../AA_references/ComplexMembrane/shortpbc.xtc'

mapping = {
    ## Resname
    "CHL1": {
    # Bead name;  Mapping.
        "ROH": {"type": "P1",    "charge": 0,  "atoms": ['C2', 'C2', 'C4', 'C4', 'O3', 'O3', 'O3', 'O3'] },      
        "R1":  {"type": "SC4",   "charge": 0,  "atoms": ['C6', 'C6', 'C6', 'C6','C6', 'C5', 'C7','H6','H7A','H7B'] },             
        "R2":  {"type": "SC3",   "charge": 0,  "atoms": ['C9', 'H9', 'C10', 'C1','H1B','H1A','C9', 'H9', 'C10', 'C1','H1B','H1A','H1B','H1A','H1A','H1B'] }, 
        "R3":  {"type": "SC3",   "charge": 0,  "atoms": ['C15', 'C15', 'C15', 'C15', 'C15', 'C14','C14','C16','C16'] },      
        "R4":  {"type": "SC3",   "charge": 0,  "atoms": ['C11', 'C12'] }, 
        "R5":  {"type": "TC2",   "charge": 0,  "atoms": ['C19'] },              
        "R6":  {"type": "TC2",   "charge": 0,  "atoms": ['C18'] },              
        "C1":  {"type": "C2",    "charge": 0,  "atoms": ['C20','H22A','H22B','H20','H21A','H21B','H21C'] },
        "C2":  {"type": "C2",    "charge": 0,  "atoms": ['C23', 'C24', 'C25', 'C26', 'C27'] },
    },
}

shaker.mapper.map_aa2cg(GRO, XTC, mapping)

In [ ]:
shaker.vsites.align_mol_to_single_traj('cg_mapped.gro', 'cg_mapped.xtc',
                    selection="resname CHOL CHL1 SITO ERG CAMP STIG",
                    align_selection="name C1 R2 R1",
                    reference_residue=0,)

In [ ]:
view = nv.show_mdanalysis(md.Universe('cg_mapped.gro','cg_mapped.xtc'))
view.clear_representations()
view.add_representation("spacefill", selection="all", radius=2.5)
view

In [ ]:
view = nv.show_mdanalysis(md.Universe('average_molecule.gro', 'aligned_molecules.xtc'))
view.clear_representations()
view.add_representation("spacefill", selection="all", radius=2.5)
view

In [ ]:
u=md.Universe('average_molecule.gro')

lines, mapping = shaker.vsites.generate_virtual_sites3(u, ['C1','R2',"R1"], output='index', selection='not name C2', 
                                         mass_split="equal", mapping=mapping, resname='CHL1')

In [ ]:
lines

In [ ]:
shaker.itp.write_initial_CGitp("CHL1", mapping,
                               header = itp_header,
                               footer=lines)